# HW3 sentiment classification

Train three fold-bagged uncased BERT models in Colab. Each encoder learns masked tokens on its own training fold before three epochs of sentiment fine-tuning. The notebook reports out-of-fold and held-out test macro-F1 and exports benchmark predictions.

In [ ]:
%%bash
set -e
cd /content

if [ -d Intro_to_AI_HW/.git ]; then
    git -C Intro_to_AI_HW pull --ff-only origin main
else
    git clone --branch main https://github.com/billlllllllly/Intro_to_AI_HW.git
fi

cd Intro_to_AI_HW/IAI_HW3
pip install -r requirements.txt

In [ ]:
%cd /content/Intro_to_AI_HW/IAI_HW3
import torch
from runpy import run_path

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training.")

run_bagged = run_path('fold_training.py')['run_bagged']
predictions, histories = run_bagged(
    checkpoint_dir='/content/hw3_fold_checkpoints',
    review_path='/content/neutral_errors.csv',
    epochs=3, mlm_epochs=2, head_lr=1.5e-4, layer_decay=0.9, smoothing=0.1,
)

In [ ]:
import pandas as pd

errors = pd.read_csv('/content/neutral_errors.csv')
print('Cross-calibrated out-of-fold errors by direction')
display(errors.groupby(['label', 'prediction']).size().rename('count').to_frame())
display(errors.groupby('label', group_keys=False).head(10))

In [ ]:
from google.colab import files

output_path = "/content/predictions.csv"
predictions.to_csv(output_path, index=False)
files.download(output_path)

In [ ]:
import matplotlib.pyplot as plt

for name, history in histories.items():
    epochs = range(1, len(history) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(9, 4))
    figure.suptitle(name)
    for index, title in enumerate(("Cross-entropy", "Macro-F1")):
        axes[index].plot(epochs, [train[index] for train, _ in history], label="Train")
        axes[index].plot(epochs, [val[index] for _, val in history], label="Validation")
        axes[index].set(xlabel="Epoch", title=title, xticks=list(epochs))
        axes[index].legend()
    figure.tight_layout()
    plt.show()